In [34]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

from typing import TypedDict, Annotated
from langchain_groq import ChatGroq

import os
from dotenv import load_dotenv

from langchain_core.messages import BaseMessage, HumanMessage

# Persistence
from langgraph.checkpoint.memory import InMemorySaver

In [35]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)


In [36]:
class jokeState(TypedDict):
    topic:str
    joke:str
    explanation:str
    

In [37]:
def generate_joke(state : jokeState):
    prompt = f"generate a joke on the topic {state["topic"]}"
    response = llm.invoke(prompt).content
    return {'joke' : response}

In [38]:
def generate_explanation(state : jokeState):
    prompt = f"generate a explanation on the joke{state["joke"]}"
    response = llm.invoke(prompt).content
    return {'explanation' : response}

In [39]:
graph = StateGraph(jokeState)
graph.add_node("generate_joke" ,generate_joke )
graph.add_node("generate_explanation" , generate_explanation )

graph.add_edge(START , "generate_joke")
graph.add_edge("generate_joke" , "generate_explanation")
graph.add_edge("generate_explanation" , END)

checkpointer = InMemorySaver()
workflow = graph.compile(checkpointer=checkpointer)

In [40]:
config1 = {"configurable": {"thread_id" : "1"}}
workflow.invoke({'topic' : 'pizza'} , config = config1)

{'topic': 'pizza',
 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty!',
 'explanation': 'A classic play on words. This joke is an example of a pun, which is a type of wordplay that exploits multiple meanings of a word or phrase. Here\'s a breakdown of how the joke works:\n\nThe joke starts by setting up a situation where a pizza is in a bad mood, which is an unexpected and humorous concept. The punchline "Because it was feeling a little crusty" is where the wordplay comes in.\n\nThe word "crusty" has a double meaning here. In one sense, a pizza crust is the outer layer of the pizza, made of bread or dough. However, "crusty" can also mean being irritable, gruff, or having a rough demeanor. This is the sense in which the joke is using the word, implying that the pizza is in a bad mood because it\'s feeling irritable or grumpy.\n\nThe humor comes from the unexpected twist on the word\'s meaning. The listener is initially thinking about the literal crust 

In [41]:
workflow.get_state(config2)

StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': '2'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())

In [42]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty!', 'explanation': 'A classic play on words. This joke is an example of a pun, which is a type of wordplay that exploits multiple meanings of a word or phrase. Here\'s a breakdown of how the joke works:\n\nThe joke starts by setting up a situation where a pizza is in a bad mood, which is an unexpected and humorous concept. The punchline "Because it was feeling a little crusty" is where the wordplay comes in.\n\nThe word "crusty" has a double meaning here. In one sense, a pizza crust is the outer layer of the pizza, made of bread or dough. However, "crusty" can also mean being irritable, gruff, or having a rough demeanor. This is the sense in which the joke is using the word, implying that the pizza is in a bad mood because it\'s feeling irritable or grumpy.\n\nThe humor comes from the unexpected twist on the word\'s meaning. The listener is initially thinking abou

In [43]:
config2 = {"configurable": {"thread_id" : "2"}}
workflow.invoke({'topic' : 'pasta'} , config = config2)

{'topic': 'pasta',
 'joke': 'Why did the spaghetti refuse to get married?\n\nBecause it was afraid of getting tangled up in a relationship!',
 'explanation': 'A classic play on words. This joke is a clever example of a pun, which relies on a double meaning of a phrase to create humor. Let\'s break it down:\n\nThe setup for the joke is "Why did the spaghetti refuse to get married?" which primes the listener to expect a reason related to relationships or commitment. The punchline "Because it was afraid of getting tangled up in a relationship" is where the wordplay comes in.\n\nThe phrase "tangled up" has a literal meaning, referring to the physical state of being entwined or knotted. However, in the context of relationships, "tangled up" is also an idiomatic expression that means to become deeply involved or complicated. The joke relies on this dual meaning to create a humorous connection between the physical properties of spaghetti (a long, thin, and easily entwined food) and the emotio